# Pairs Trading: does statistical arbitrage survive out of sample?

**A walk-forward study of cointegration-based mean reversion on 25 large-cap US equities and sector ETFs, 2012–2026.**

---

This notebook is the **narrative** of the study. It computes nothing new: every number
below is read from the CSVs in `data/`, which are produced by the scripts in `src/`.
That split is deliberate —

* `src/*.py` do the work, print their findings to stdout, and write their results to `data/`.
  Scripts are reproducible and diffable: `python src/backtest.py > log`, commit the log,
  and you can see exactly what changed when the code changed.
* this notebook does the *explaining*, and can never disagree with the scripts because it
  has no independent copy of the logic.

If you want the mechanics, read the module docstrings in `src/` — they are written to
teach. If you want the argument, read on.

---

## The research question

> Do cointegration-selected equity pairs generate out-of-sample, cost-aware,
> risk-adjusted returns — and if the in-sample numbers look good, how much of that is
> real and how much is the multiple-testing problem?

Note the shape of that question. It is not "build a profitable pairs strategy." It has a
**publishable answer either way**, which is the only kind of question worth spending a
month on. The finding is negative, and the value of the project is the machinery that
establishes the negative result *credibly* — because the same machinery is what would let
you believe a positive one.

## The answer, up front

| | |
|---|---|
| Pairs tested (full sample) | 435 |
| Cointegrated at p < 0.05 | 23 — against **21.75 expected by chance alone** |
| Passing the full filter | **0** |
| Walk-forward OOS Sharpe (5 bp/side) | **−0.11** over 3,150 trading days |
| Mean in-sample Sharpe of selected pairs | **+1.54** |
| Mean out-of-sample Sharpe of the same pairs | **−0.11** |
| Cointegration that survives into the trading window | **3.3%** |
| Sharpe at *zero* transaction cost | −0.05 (it does not lose to costs; it has no edge) |
| Deflated Sharpe Ratio | **0.00%** |

Everything below is the argument that those numbers mean what I say they mean.

In [ ]:
# Setup: this notebook reads results only. No model is re-fit here.
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, FIGS = ROOT / "data", ROOT / "figures"

prices  = pd.read_csv(DATA / "prices.csv", index_col=0, parse_dates=True)
screen  = pd.read_csv(DATA / "pair_screen_full_sample.csv")
rolls   = pd.read_csv(DATA / "roll_records.csv")
pairs   = pd.read_csv(DATA / "pair_records.csv")
trades  = pd.read_csv(DATA / "trades.csv")
oos     = pd.read_csv(DATA / "oos_returns.csv", index_col=0, parse_dates=True)["net_return"]
costs   = pd.read_csv(DATA / "cost_sensitivity.csv")
grid    = pd.read_csv(DATA / "parameter_sensitivity.csv")

print(f"prices    {prices.shape[0]:>5} days x {prices.shape[1]} tickers   "
      f"{prices.index[0].date()} -> {prices.index[-1].date()}")
print(f"screen    {len(screen):>5} pairs")
print(f"rolls     {len(rolls):>5} walk-forward windows")
print(f"pairs     {len(pairs):>5} (roll, pair) selections")
print(f"trades    {len(trades):>5} round trips")
print(f"oos       {len(oos):>5} out-of-sample days")

---

## 1. What pairs trading is, in one paragraph

Two assets whose prices are driven by the same economic forces should not drift apart
forever. If Coca-Cola and PepsiCo both sell sugar water into the same demand shocks and
face the same input costs, the *ratio* of their prices should be a stable quantity even
though each price individually wanders. So construct a portfolio — long one, short a
hedge ratio of the other — whose value is that stable quantity, the **spread**. When the
spread is unusually high, sell it; when unusually low, buy it; when it returns to normal,
close. You are not forecasting either stock. You are betting that a *relationship* holds.

The attraction is that the market direction cancels: if both names fall 10%, the spread
barely moves. That makes the return stream close to market-neutral, which is why it earns
a place in a portfolio even at a modest Sharpe. The catch — and this is the whole study —
is that "the relationship holds" is a statistical claim, and statistical claims found by
searching hundreds of candidates are usually claims about the search, not the world.

## 2. Cointegration, not correlation

The single most common error in this space is to screen on correlation. Correlation is a
statement about *co-movement of returns*; cointegration is a statement about *the levels
not drifting apart*. They are different, and only the second one is tradeable.

The standard picture: two drunks stumbling home from a bar. Their step-to-step movements
are perfectly correlated with nothing — each step is random. But if they are chained
together at the wrist, the *distance* between them cannot grow without bound. That leash
is cointegration. Correlated returns with no leash means the gap can widen forever, and
your mean-reversion bet has no mean to revert to.

Formally: each price series is I(1) — it has a unit root, its first difference is
stationary, and its variance grows with time. A pair is cointegrated if some linear
combination $\log P_A - \beta \log P_B$ is I(0), i.e. stationary: it has a fixed mean and
a fixed variance, so "unusually far from the mean" is a meaningful phrase. Without that,
the z-score you compute is a number divided by a standard deviation that does not exist.

**Why log prices for the relationship, simple returns for the P&L.** The hedge ratio from
a regression of log prices is an elasticity: a 1% move in B goes with a β% move in A,
which is scale-free and stable as prices grow. But log returns do not aggregate across a
portfolio — the return of a weighted book is the weighted sum of *simple* returns, not of
log returns. Using log returns to compute P&L introduces a small, systematic, always-
downward error. So: logs for the econometrics, simple returns for the money. Both live in
`src/data_pipeline.py`, deliberately as two separate functions.

In [ ]:
# The KO-PEP cautionary tale: the textbook pair, in the data, is not cointegrated.
example = screen[screen.pair.isin(["KO-PEP", "LOW-PEP", "BAC-JPM"])]
cols = ["pair", "same_sector", "beta", "eg_pvalue", "half_life_days", "selected"]
display(example[cols].sort_values("eg_pvalue").reset_index(drop=True))

# ...while their *returns* are strongly correlated. Correlation is not the leash.
rets = prices.pct_change().dropna()
print(f"\ncorr(KO, PEP) daily returns : {rets['KO'].corr(rets['PEP']):+.3f}")
print(f"KO-PEP Engle-Granger p-value : "
      f"{float(screen.loc[screen.pair == 'KO-PEP', 'eg_pvalue'].iloc[0]):.3f}")
print("\nHighly correlated. Not cointegrated. Untradeable as a spread.")

---

## 3. The screen, and the number that decided the project

`src/cointegration.py` tests all pairs in the panel: 25 universe names plus the 5
positive-control ETFs, so `C(30,2) = 435` candidate pairs (see `candidate_pairs()`). Five
filters:

1. **Leg gate** — both legs must fail to reject a unit root at α = 0.01. A pair of already-
   stationary series can produce a "cointegrated" verdict trivially and meaninglessly.
   (α = 0.01 not 0.05: this gate's job is to exclude only the clear cases, and its
   empirical size is measured on synthetic random walks in `tests/test_pairs.py`.)
2. **Engle–Granger test** via `statsmodels.tsa.stattools.coint`, at p < 0.05. Not
   `adfuller()` on OLS residuals — that is the mistake that inflates significance,
   because the residuals are *fitted* to be small, so the standard Dickey–Fuller critical
   values are wrong. `coint` uses the Phillips–Ouliaris/MacKinnon critical values that
   account for the estimated cointegrating vector.
3. **Half-life band**, 5–60 trading days. From the AR(1) fit
   $\Delta s_t = a + b\, s_{t-1}$, with $\phi = 1 + b$ and
   $\text{HL} = -\ln 2 / \ln \phi$. Too fast and the reversion is inside the bid-ask
   spread; too slow and capital is tied up for a year per trade with a structural break
   waiting to happen.
4. **Johansen trace test** as a symmetric cross-check — Engle–Granger depends on which
   leg you put on the left-hand side, and Johansen does not.

Here is the result that set the tone for everything after it.

In [ ]:
n = len(screen)
sig = int((screen.eg_pvalue < 0.05).sum())
print(f"pairs tested                       : {n}")
print(f"significant at p < 0.05            : {sig}")
print(f"EXPECTED BY CHANCE if no pair is   : {n * 0.05:.2f}")
print(f"selected after the full filter     : {int(screen.selected.sum())}")

# Why nothing survived: every significant pair reverts far too slowly to trade.
hl = screen.loc[screen.eg_pvalue < 0.05, "half_life_days"]
print(f"\nhalf-life of the {sig} significant pairs: "
      f"min {hl.min():.0f}d, median {hl.median():.0f}d, max {hl.max():.0f}d")
print("filter band is 5-60 days -> 0 pass")

# And the direction of the effect is backwards from the economic story.
by_sector = (screen.assign(hit=screen.eg_pvalue < 0.05)
                   .groupby("same_sector")["hit"].agg(["sum", "count", "mean"]))
by_sector.index = ["cross-sector", "same-sector"]
display(by_sector.rename(columns={"sum": "significant", "count": "tested", "mean": "hit_rate"}))

### Read that carefully

**23 significant, 21.75 expected by chance.** If not one pair in the universe were truly
cointegrated, a 5%-level test run 435 times would flag about 22 of them anyway. I found
23. The entire signal in my screen is consistent with pure noise, and I know that before
running a single backtest.

**And the economics point the wrong way.** Same-sector pairs — the ones with an actual
story about shared demand shocks and shared input costs — hit at **2.9%** (1 of 34).
Cross-sector pairs, where no such story exists, hit at **5.5%** (22 of 401). If the hits
were finding real economic links, that ordering would be reversed. Instead the
cross-sector rate is almost exactly the nominal size of the test, which is what a
distribution of p-values under the null looks like.

**Zero pairs survived the filter, and that is not a bug.** All 23 failed the half-life
band — the fastest reverted in 64 days, the median in 102. A 100-day half-life means
that after five months, half the deviation is still there. That is not mean reversion you
can trade; it is a slow-moving relationship in which "the relationship broke" and "it is
still reverting" are observationally identical for the better part of a year. I checked
this was a filter and not a crash by loosening the band and watching pairs appear.

The smallest p-value in the entire 435-pair screen is **0.0043** (LOW–PEP: Lowe's and
PepsiCo — a hardware retailer and a soft-drink maker, which should tell you what it is).
Under Bonferroni, 435 tests need p < 0.000115 to claim significance at a family-wise 5%.
Nothing in the screen is within a factor of thirty of that bar.

---

## 4. The backtest that could not cheat

Zero pairs surviving on the full sample is a finding, not a study. The interesting
question is what happens to someone who *does* trade this — who re-screens periodically,
takes the best pairs available, and trades them forward. So `src/backtest.py` runs a
walk-forward with the selection **inside** the loop:

```
[ 12-month formation ][ 6-month trading ]
                      [ 12-month formation ][ 6-month trading ]
                                            [ 12-month formation ][ 6-month trading ]
```

The 12/6 convention follows Gatev, Goetzmann & Rouwenhorst (2006), which matters because
it means the window lengths are not one of my tunable knobs.

Four design decisions do the real work here:

1. **Pair selection happens inside each formation window.** This is the difference between
   a walk-forward and a fig leaf. If you pick the pairs once on the full sample and then
   "walk forward" the trading, you have leaked the answer: the pairs were chosen *because*
   they mean-reverted over a period that includes your test set. The top-5 pairs
   differ from roll to roll here, and that churn is itself a result (see §7).
2. **Formation parameters are frozen.** `α`, `β`, the spread mean and the spread sd are
   computed on the formation window and stored in a `@dataclass(frozen=True)`
   (`src/signals.py`). The trading-window z-score is
   $(s_t - \mu_{\text{formation}}) / \sigma_{\text{formation}}$ — never
   re-standardised on data the strategy has not lived through. This is what makes the
   z-score genuinely out-of-sample rather than out-of-sample-shaped.
3. **`.shift(1)` on every position.** The signal computed from the close of day *t* earns
   the return of day *t+1*. Without it, you trade on a price you learned at the moment
   you traded, which is worth an enormous fake Sharpe.
4. **Test windows do not overlap.** Overlapping test periods reuse the same days across
   rolls, which understates the variance of every statistic computed from them.

The two tamper tests in `tests/test_pairs.py` are the only real defence against lookahead:
vandalise the last 50 days of the price series, re-run, and assert every earlier net
return is byte-identical. A leak anywhere in the chain fails that test.

In [ ]:
from scipy import stats as sstats   # same definitions as src/backtest.py

ANN = 252


def describe(r: pd.Series, benchmark: pd.Series | None = None) -> list[float]:
    """The same summary statistics src/backtest.py reports, computed the same way.

    Two definitions are worth spelling out because they are easy to get wrong:

    * HIT RATE is over ACTIVE days only (r != 0). Counting flat days as losses
      would drag it toward zero and say nothing about the strategy's accuracy;
      counting them as wins would be worse. Only days with risk on are informative.
    * KURTOSIS is the RAW fourth moment (scipy fisher=False), so 3.0 is normal,
      not 0.0. Stated because "kurtosis = 17" and "excess kurtosis = 17" are very
      different claims and nobody can tell which you mean from the number alone.
    """
    equity = (1 + r).cumprod()
    years = len(r) / ANN
    active = r[r != 0]
    return [
        len(r),
        equity.iloc[-1] - 1,
        equity.iloc[-1] ** (1 / years) - 1,
        r.std() * np.sqrt(ANN),
        r.mean() / r.std() * np.sqrt(ANN),
        (equity / equity.cummax() - 1).min(),
        (active > 0).mean(),
        sstats.skew(r),
        sstats.kurtosis(r, fisher=False),
        r.corr(benchmark) if benchmark is not None else 1.0,
    ]


spy = prices["SPY"].pct_change().reindex(oos.index).fillna(0)
strategy = oos.fillna(0)

summary = pd.DataFrame(
    {"pairs strategy (net 5bp)": describe(strategy, benchmark=spy),
     "SPY buy-and-hold (same days)": describe(spy)},
    index=["OOS days", "total return", "ann. return", "ann. vol", "Sharpe",
           "max drawdown", "hit rate (active days)", "skew",
           "kurtosis (3 = normal)", "corr to SPY"])
display(summary.round(4))

### What that table says, and what it does not

The strategy loses money: **−0.56% a year, Sharpe −0.11**, over 3,150 out-of-sample days.
No amount of framing makes that an edge.

But three of those rows are worth more than the headline:

* **Annualised vol 4.15%.** The hedging works. This is a market-neutral return stream in
  the mechanical sense — one-fifth the volatility of the index over the same days.
* **Correlation to SPY +0.04.** So does the neutrality. A strategy with a genuine edge and
  this correlation profile would be extremely valuable *precisely because* the Sharpe
  does not have to be large to earn allocation. Which is exactly why the multiple-testing
  discipline in §8 matters so much: the payoff to a false positive here is enormous, so
  the incentive to fool yourself is enormous.
* **Kurtosis 17.3** (raw; 3.0 would be normal). Fat tails, from the days when a spread gaps and the stop
  fires. Reported vol understates the real risk; a vol-based position sizer would be
  systematically too aggressive.

Note also the max drawdown comparison: −17.9% for the strategy against −33.7% for SPY.
This is the trap in market-neutral reporting. A smaller drawdown on a strategy that
*earns nothing* is not risk management, it is a smaller version of nothing. Drawdown is
only meaningful next to a return.

In [ ]:
display(Image(filename=str(FIGS / "equity_curve.png")))

---

## 5. The anatomy of "no edge"

An aggregate Sharpe of −0.11 could hide many different diseases: a strategy that wins
often and loses catastrophically, one that never triggers, one killed entirely by costs.
The trade-level table distinguishes them, and here the diagnosis is the most boring
possible one.

In [ ]:
by_reason = trades.groupby("exit_reason").agg(
    n=("pnl", "size"), mean_pnl=("pnl", "mean"), mean_days=("days", "mean"))
by_reason["share"] = by_reason.n / len(trades)
display(by_reason.sort_values("n", ascending=False))

wins, losses = trades.pnl[trades.pnl > 0], trades.pnl[trades.pnl <= 0]
print(f"round trips        : {len(trades)}")
print(f"win rate           : {len(wins) / len(trades):.1%}")
print(f"mean win           : {wins.mean():+.2%}")
print(f"mean loss          : {losses.mean():+.2%}")
print(f"win/loss ratio     : {abs(wins.mean() / losses.mean()):.2f}")
print(f"mean holding period: {trades.days.mean():.1f} trading days")
print(f"mean P&L per trade : {trades.pnl.mean():+.2%}   <-- the whole story")

**A 50.8% win rate with a win/loss ratio of 0.96.** Mean win +3.12%, mean loss −3.25%,
mean P&L per trade −0.02%. This is a coin flip that pays slightly less than it costs — a
fair game minus friction. It is the cleanest possible picture of *no edge*, and it rules
out the alternative diagnoses:

* Not a **frequency** problem: 195 round trips, 35% of days in the market. Plenty of
  chances to express the edge, if there were one.
* Not a **cost** problem: see §6 — the Sharpe is still negative at zero cost.
* Not a **tail** problem: the target and stop exits are near-symmetric (+4.01% on 69
  target exits, −4.03% on 62 stops). The stop is doing its job, cutting losses at about
  the size of the wins rather than letting them run.
* The 42 `window_end` exits (−0.30% average) are the honest cost of the walk-forward: the
  trading window closes and open positions are liquidated regardless of the z-score. A
  production system would roll them into the next window's parameters. I chose not to,
  because doing so would let a position span two formation regimes and blur what the
  out-of-sample test is measuring.

In [ ]:
display(Image(filename=str(FIGS / "spread_with_trades.png")))

The example pair above is chosen by **most trades**, not best Sharpe. Picking the best
pair would make this figure marketing. A criterion independent of profitability makes it
the most informative chart available instead of the most flattering one, and you can see
the actual failure mode in it: entries at ±2σ that revert cleanly and pay, alongside the
ones that keep going and hit the stop.

---

## 6. Costs: the strategy does not die of friction, it was never alive

The cost model in `src/backtest.py` charges a one-way cost in basis points on **turnover**
— the change in gross exposure, not the position level — plus a borrow charge on the short
leg. Charging costs on the position rather than on turnover is a common and expensive
error: it makes a position held for 30 days look 30 times more expensive than it is,
which flatters high-frequency variants and penalises the ones you can actually trade.

Because every row below reuses the *same* cached walk-forward plan (same pairs, same
hedge ratios, same z-score paths), these five rows differ **only** in cost. That is what
a sensitivity analysis is supposed to mean and it is easy to get wrong: re-screening for
each cost level would let the pair selection change underneath you, and then the table
measures selection noise, not cost.

In [ ]:
display(costs.set_index("one_way_bps").round(4))
display(Image(filename=str(FIGS / "cost_sensitivity.png")))
print("Sharpe at ZERO transaction cost:",
      f"{float(costs.loc[costs.one_way_bps == 0, 'sharpe'].iloc[0]):+.3f}")

**The break-even cost is negative.** At 0 bp the Sharpe is −0.05; at my 5 bp headline
assumption, −0.11; at 20 bp, −0.32. The slope tells you costs matter — roughly 0.014
Sharpe per basis point — but the intercept tells you the important thing: **there is no
gross edge for costs to eat.** If the zero-cost Sharpe had been +0.9 and the 5 bp Sharpe
−0.11, this would be a very different project: an implementation problem, worth attacking
with better execution. It is not. It is an absence-of-signal problem.

This is why I report the zero-cost row at all. It separates "my alpha is real but my
costs are too high" from "I have no alpha," and those two findings lead to completely
different next steps.

---

## 7. The finding: in-sample Sharpe carries no information

Now the number the project exists for. For every one of the 122 (roll, pair) selections I
have the in-sample Sharpe from the formation window that *caused* the pair to be selected,
and — for the 111 that actually traded — the out-of-sample Sharpe from the trading window
that followed it.

In [ ]:
# Aggregates use all 122 selections. The 11 pairs that never traded in their
# window have an undefined OOS Sharpe (zero-variance return series -> NaN), and
# pandas skips them; the correlation needs both legs, so it uses the other 111.
# Dropping those 11 everywhere would silently discard "the signal never fired",
# which is itself an outcome.
paired = pairs.dropna(subset=["is_sharpe", "oos_sharpe"])

print(f"selections                     : {len(pairs)}  "
      f"({int(pairs.oos_n_trades.eq(0).sum())} never traded -> OOS Sharpe undefined)")
print(f"mean IS  Sharpe                : {pairs.is_sharpe.mean():+.2f}")
print(f"mean OOS Sharpe                : {pairs.oos_sharpe.mean():+.2f}")
print(f"decay                          : "
      f"{pairs.is_sharpe.mean() - pairs.oos_sharpe.mean():.2f} Sharpe points")
print(f"corr(IS, OOS) Pearson  (n={len(paired)}) : "
      f"{paired.is_sharpe.corr(paired.oos_sharpe):+.3f}")
print(f"corr(IS, OOS) Spearman (n={len(paired)}) : "
      f"{paired.is_sharpe.corr(paired.oos_sharpe, method='spearman'):+.3f}")
print(f"\nOOS Sharpe > 0                 : {(pairs.oos_sharpe > 0).mean():.1%} of the "
      f"{int(pairs.oos_sharpe.notna().sum())} that traded")
print(f"cointegration STILL significant in the trading window: "
      f"{pairs.cointegration_survived.mean():.1%}")
display(Image(filename=str(FIGS / "is_vs_oos_sharpe.png")))

### 1.54 → −0.11

Every pair enters the trading window with an average formation Sharpe of **+1.54**. On a
one-year window, that looks like a strategy. It comes out at **−0.11**. The scatter plot
is the argument in one picture: if the screen had predictive content, the dots would lie
along the 45° line. They spread horizontally instead — the IS Sharpe tells you nothing
about the OOS Sharpe. **That gap is not bad luck. It is the definition of selection bias**:
I chose these pairs *because* their formation Sharpe was high, and the component of a
high Sharpe that is noise does not repeat.

And the mechanism is visible one level down: **only 3.3% of selected pairs are still
cointegrated at p < 0.05 in their own trading window.** The property the entire strategy
is built on does not survive six months, in 97% of the cases where a 12-month test said
it was there. Which is exactly what you would expect if the 12-month test was mostly
finding noise — as §3's 23-vs-21.75 already suggested.

This is also the answer to a tempting patch: "use a tighter p-value threshold." It would
not help. In this data, formation p-value has a Spearman correlation of **−0.04** with
OOS Sharpe. Ranking candidates by statistical significance is ranking them by nothing.

---

## 8. The audit: how many things did I try?

Every result above is a *reported* number, and reported numbers are selected numbers. The
honest accounting of what a study searched over is `src/multiple_testing.py`.

In [ ]:
n_pairs, n_rolls, n_cells = len(screen), len(rolls), len(grid)
N = n_pairs * n_rolls * n_cells

print("THE SIZE OF THE SEARCH")
print(f"  pairs screened per formation window : {n_pairs}")
print(f"  walk-forward rolls                  : {n_rolls}")
print(f"  threshold combinations reported     : {n_cells}")
print(f"  => effective strategy trials N      : {N:,}   (the DSR's N)")

# TWO DIFFERENT FAMILIES, and conflating them is a real mistake:
#   - Bonferroni/BH here correct the 435 p-values of ONE screen, so the family
#     size is 435 and the threshold is 0.05/435.
#   - The DSR asks a different question -- "how good would the BEST of all my
#     strategy trials look by luck?" -- so its N is the full 195,750.
print("\nMULTIPLICITY CORRECTION ON THE FULL-SAMPLE SCREEN (family = 435 tests)")
print(f"  Bonferroni threshold at FWER 5%     : {0.05 / n_pairs:.3e}")
print(f"  smallest p-value in the screen      : {screen.eg_pvalue.min():.4f}"
      f"   ({screen.loc[screen.eg_pvalue.idxmin(), 'pair']})")
print(f"  ratio, smallest p to the threshold  : "
      f"{screen.eg_pvalue.min() / (0.05 / n_pairs):.0f}x too large")
print("  survivors, Bonferroni               : 0")
print("  survivors, Benjamini-Hochberg FDR   : 0")

`N = 435 × 25 × 18 = 195,750`. That is the number I have to defend, and I would rather
state it than have an interviewer find it.

* **Bonferroni** (family-wise error) on the 435 p-values of one screen: threshold
  `0.05 / 435 = 1.15e-4`, and the smallest p-value in the whole screen (0.0043) misses it
  by a factor of 37. **Zero survivors.**
* **Benjamini–Hochberg** (false discovery rate, the more appropriate tool here since the
  tests are correlated and I care about the *proportion* of my discoveries that are false,
  not about never making one error): **zero survivors** at q = 0.05.
* **Probabilistic Sharpe Ratio** — the probability the true Sharpe exceeds zero, given the
  observed Sharpe, the sample length, and the return distribution's skew and kurtosis:
  **34.3%**. Not "the strategy is bad" but "I cannot tell, and what I can tell leans
  negative."
* **Deflated Sharpe Ratio** — the PSR against the Sharpe you would *expect* the best of N
  trials to produce by luck alone: **0.00%.**

The DSR uses the Bailey–López de Prado expected-maximum result,
$\mathbb{E}[\max SR_N] \approx \sigma_{SR}\left[(1-\gamma)\Phi^{-1}(1-1/N) + \gamma\,\Phi^{-1}(1-1/(Ne))\right]$
with $\gamma$ = 0.5772 (Euler–Mascheroni). With $\sigma_{SR}$ = 1.76 across trials, the
bar comes out at **E[max SR] = 7.97**.

**And I will criticise my own number before anyone else does.** A 7.97 Sharpe bar is
implausibly high, and the reason is that I estimated $\sigma_{SR}$ from single-pair,
126-day Sharpes, which are extremely noisy — that variance inflates the deflation bar far
beyond anything the trials really justify. The honest version estimates $\sigma_{SR}$
across *strategy-level* trials, not pair-level ones. It does not change the conclusion
here (DSR is 0.00% at a bar of 2 as easily as at 7.97), but if my Sharpe had been +1.3 it
would have been the difference between a false negative and a real result. I would rather
report the flaw than the flattering version.

In [ ]:
# The threshold grid: reported as a diagnostic, never used to select.
g = grid.sort_values("sharpe", ascending=False)
display(g.reset_index(drop=True).round(4))
print(f"cells                : {len(grid)}")
print(f"Sharpe range         : {grid.sharpe.min():+.2f} to {grid.sharpe.max():+.2f}")
print(f"negative Sharpe cells: {(grid.sharpe < 0).sum()} of {len(grid)}")

The headline configuration — entry 2.0σ, exit 0.0σ, stop 3.0σ — was **pre-committed**,
chosen from the literature before any of these numbers existed, and it lands at −0.11, in
the *middle* of the grid. The best of eighteen cells is +0.06, statistically
indistinguishable from zero; 16 of 18 are negative.

That is what a robustly absent edge looks like, and it forecloses the obvious objection
("you just picked bad parameters"): there is no lucky cell to find. Had I reported the
best cell as my result, +0.06 would have been an overfit to eighteen tries — and the
grid's own existence is why `n_parameter_combinations=18` is an input to the DSR above.

---

## 9. The positive controls, and why they failed

Before trusting a pipeline that says "no," you have to know it can say "yes." So I ran
five pairs that *should* be cointegrated by construction — ETFs tracking the same index
(GLD/IAU, VOO/SPY, and similar).

Several failed. Two reasons, and both are real economics rather than bugs:

1. **Expense-ratio differential.** GLD charges 0.40%/yr and IAU 0.25%. Two funds holding
   the same metal but bleeding fees at different rates have a spread with a
   **deterministic drift** of about 15 bp a year. A drifting spread is not stationary,
   so a test whose null is a unit root fails to reject — correctly. The pair is not
   cointegrated; it is cointegrated-around-a-trend, which is a different model.
2. **A structural break in 2016.** Perron (1989): unit-root tests have low power against
   a structural break. A single level shift in an otherwise stationary series looks
   exactly like a random walk to an ADF test. This is a limitation of the *tool*, and it
   matters for the main result too — some of the pairs I rejected may be stationary
   around a break I did not model.

There is a subtlety worth stating because it confused me for an hour: the OLS β partially
*absorbs* the drift, so the measured drift in the fitted spread is smaller than the
15 bp/yr the expense ratios imply. The regression trades a little bias in β for a smaller
residual, which is what least squares does — and it means you cannot read the economic
drift straight off the residual series.

The failing controls are in the repo, failing. A positive control that fails and gets
quietly deleted is worse than no control at all.

---

## 10. What I got wrong

Four things, all of which taught me more than the successes:

1. **The stop-loss re-entry bug.** My first state machine, on a stop-out, closed the
   position while the z-score was still beyond the entry threshold — so the next bar it
   re-entered the same losing trade, and kept doing so until the spread came back. The
   backtest still "worked": no crash, no NaN, just a plausible-looking Sharpe. Fixed with
   a `disarmed` flag that requires the z-score to return inside the band before the pair
   can trade again. **Lesson: a backtest cannot tell you your logic is wrong, only that
   it ran.** The trade log found it; the summary statistics never would have.
2. **`adfuller` on OLS residuals.** My first Engle–Granger implementation used standard
   Dickey–Fuller critical values on fitted residuals, which are biased toward
   stationarity because the regression *chose* β to make them small. It reported far more
   cointegration than exists. Replaced with `statsmodels.tsa.stattools.coint`.
3. **A finite half-life on a pure random walk.** I panicked at this in a unit test before
   recognising Dickey–Fuller bias: $\hat\phi$ is biased *downward* in finite samples, so a
   true random walk (φ = 1, infinite half-life) estimates φ < 1 and yields a finite
   half-life. Not a bug — a known property, and one that biases my half-life filter toward
   admitting pairs that do not revert at all.
4. **The plan/execute split, learned the hard way.** The first version re-screened all 435
   pairs inside every backtest variation, so the cost sensitivity (5 runs) and threshold
   grid (18 runs) each meant re-doing a 13-minute screen. Refactoring into "build the plan
   once, then compute P&L many times" made 17 variations feasible — and, more importantly,
   *guarantees* that a sensitivity table's rows differ only in the parameter being varied.
   The cache then produced its own bug: pickling the dataclasses recorded them as
   `__main__.RollPlan` (because `backtest.py` was run as a script), so loading the cache
   from `multiple_testing.py` failed. It now stores plain dicts and pandas objects. **A
   cache file that depends on how the writer was invoked is not a cache, it is a trap.**

---

## 11. Limitations I would fix first

Ordered by how much they threaten the conclusion:

1. **Survivorship bias.** The universe is 25 names that exist *today*. Every company that
   was delisted, acquired or went bankrupt is missing, and those are exactly the cases
   where a spread diverges and never comes back. The bias runs **upward** — my −0.11 is,
   if anything, optimistic. Unfixable with free data; it needs a point-in-time universe.
   This is the one that would make a *positive* result untrustworthy, which is why it is
   first.
2. **No structural-break modelling.** Per §9, ADF has low power against breaks, so some
   rejected pairs may be stationary around a shift. Gregory–Hansen would test for
   cointegration with a break.
3. **A 25-name universe is small.** 435 pairs sounds like a lot but the real search space
   in industry is thousands of names; more pairs means more genuine candidates *and* a
   larger multiple-testing burden, and it is not obvious which effect dominates.
4. **Static hedge ratio.** β is fixed per formation window. A Kalman filter would let it
   evolve — at the cost of new parameters that must be counted in the trial budget.
5. **Simplistic cost model.** A flat bps figure ignores that spreads widen exactly when
   pairs diverge, i.e. costs are highest precisely when the strategy wants to trade.
6. **No short-availability model.** Borrow is a flat charge; hard-to-borrow names and
   recalls are not modelled.
7. **Daily closes only.** Intraday execution, opening auctions and overnight gaps are
   invisible at this frequency.
8. **Equal weighting across pairs.** No risk parity, no correlation-aware sizing across
   the 5 concurrent pairs — so the book's diversification is accidental rather than
   designed.

---

## 12. Conclusion

The strategy does not work on this universe, at this frequency, in this period. The
evidence:

* the screen finds **23** cointegrated pairs where chance predicts **21.75**;
* same-sector pairs — the economically motivated ones — hit *less* often than
  cross-sector pairs;
* formation Sharpe **+1.54** becomes out-of-sample Sharpe **−0.11**, with IS/OOS
  correlation near zero;
* cointegration survives into the trading window **3.3%** of the time;
* the Sharpe is negative even at **zero cost**, so this is absence of signal rather than
  a cost problem;
* **16 of 18** threshold configurations lose money, and the best is +0.06;
* after accounting for **195,750** trials, the Deflated Sharpe Ratio is **0.00%**.

What I would want a reader to take from it is not the negative result — plenty of things
do not work. It is that the apparatus reached the negative result *on purpose*: frozen
formation parameters, selection inside the walk-forward, tamper tests for lookahead, a
zero-cost row to separate implementation from signal, positive controls left in the repo
failing, and an explicit trial count feeding a deflated Sharpe. Every one of those is a
place where I could have made the result look better by being slightly less careful.

The point of the machinery is that if it *had* said +1.2, I would have had a reason to
believe it.

---

### Where to go next in this repo

| | |
|---|---|
| `README.md` | the full write-up: method, results, design justifications |
| `src/cointegration.py` | the econometrics, with the reasoning in the docstrings |
| `src/signals.py` | frozen formation stats and the trade state machine |
| `src/backtest.py` | the P&L engine, walk-forward, and the plan/execute split |
| `src/multiple_testing.py` | Bonferroni, BH-FDR, PSR, DSR |
| `tests/test_pairs.py` | 27 tests — read the tamper tests first |

**Companion project:** `~/vol-forecasting-study` — volatility forecasting and options
(GARCH, realised vol, variance risk premium). Together the two cover the derivatives side
and the equity relative-value side of a quant analyst's toolkit.